In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics.pairwise import cosine_similarity

import json

In [2]:
from sentence_transformers import SentenceTransformer

model =SentenceTransformer("all-MiniLM-L6-v2",device='cuda')


/home/vighnesh/projects/coursera/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [ ]:
movies_df = pd.read_csv('../dataset/processed/movies_df_clean.csv')

In [ ]:
# import mapped lookup file which mapped movielense and tmdb movies ids of same movie.
movieId_lookup = pd.read_csv('../dataset/processed/movieId_lookup.csv')

In [ ]:
movieId_lookup.shape

In [ ]:
#  filter movies which have rated by users. 
movies_df = movies_df[movies_df['movie_id'].isin(movieId_lookup['tmdb_movieId'])].reset_index(drop=True)

In [ ]:
movies_df['overview_clean'].isna().sum()

In [ ]:
movies_df.isna().sum()

In [ ]:
def combine_all(x):
    text = f'''Genres : {x['genres_clean']}.
Keywords : {x['keywords_clean']}.
{x['crew_clean']}.
Cast : {x['cast_clean']}.
Overview :  {x['overview_clean'] }
'''.strip()

    return text
# Tagline is "{x['tagline'][:-1]}".

In [ ]:
movies_df['combined_text'] =  movies_df.apply(combine_all ,axis=1)

In [ ]:
print(movies_df['combined_text'][0])

### Build Model !!

In [ ]:
embedding = model.encode(
    inputs=movies_df['combined_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

In [ ]:
embedding.shape

In [ ]:
movies_df[movies_df['title']=='Interstellar'].index[0]

In [ ]:
# verify the embedding
movie_idx = movies_df[movies_df['title']=='Avatar'].index[0]
movie_idx

In [ ]:
# compute Similarities 
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    [embedding[movie_idx]],embedding
)[0]

In [ ]:
top_indicies = similarities.argsort()[::-1][:11]

In [ ]:
for idx in top_indicies:
    print(movies_df.loc[idx,'title'], similarities[idx])

### 3. composite model

In [ ]:
genre_emb = model.encode(inputs=movies_df['genres_clean'].tolist(),show_progress_bar=True,
                         batch_size=64, convert_to_numpy=True)
overview_emb = model.encode(inputs=movies_df['overview_clean'].tolist(),show_progress_bar=True,
                            batch_size=64,convert_to_numpy=True)
keyword_emb = model.encode(inputs=movies_df['keywords_clean'].tolist(),show_progress_bar=True,
                           batch_size=64, convert_to_numpy=True)
director_emb = model.encode(inputs=movies_df['crew_clean'].tolist(),show_progress_bar=True,
                            batch_size=64, convert_to_numpy=True)

In [ ]:
# compute Similarities 

genre_similarities = cosine_similarity(
    [genre_emb[movie_idx]],genre_emb
)[0]

overview_similarities = cosine_similarity(
    [overview_emb[movie_idx]],overview_emb
)[0]

keyword_similarities = cosine_similarity(
    [keyword_emb[movie_idx]],keyword_emb
)[0]

director_similarities = cosine_similarity(
    [director_emb[movie_idx]],director_emb
)[0]

In [ ]:
final_emb = (
    0.30 * genre_emb +
    0.30 * keyword_emb +
    0.35 * overview_emb +
    0.05 * director_emb
)


In [28]:
EMBEDDING_DIR_PATH = os.path.join('..','artifact','embeddings')
def save_embedding(file_name,emdd):
    try:
        emdd_path =os.path.join(EMBEDDING_DIR_PATH,file_name)
        np.save(emdd_path,arr=emdd)
        print(f'{file_name} saved successfull!')
    except Exception as e:
        print(f"Error process stop : {e}")
        

    
    

# save final embedded and other embedded arrays
save_embedding('final_embeddings.npy',final_emb)
save_embedding('genre_embeddings.npy',genre_emb)
save_embedding('overview_embeddings.npy',overview_emb)
save_embedding('keyword_embeddings.npy',keyword_emb)
save_embedding('director_embeddings.npy',director_emb)

final_embeddings.npy saved successfull!
genre_embeddings.npy saved successfull!
overview_embeddings.npy saved successfull!
keyword_embeddings.npy saved successfull!
director_embeddings.npy saved successfull!


In [ ]:
movies_df[movies_df['movie_id']==19995]

### Demo of process

In [ ]:
similarities = cosine_similarity(
    [final_emb[0]],final_emb
)[0]

In [60]:
top_indicies = similarities.argsort()[::-1][:11]

In [61]:
for idx in top_indicies:
    print(movies_df.loc[idx,'title'], similarities[idx])

Avatar 1.0
X-Men: The Last Stand 0.767058
Ben-Hur 0.7523298
Angels & Demons 0.74764144
DodgeBall: A True Underdog Story 0.7437382
Top Cat Begins 0.74205697
Contagion 0.741151
Batman Begins 0.72883415
Hulk 0.7276186
The Break-Up 0.72720194
Star Trek 0.7268152


In [58]:
def get_similarity_explaination(movie_idx,remm_idx):
    gen_sim = cosine_similarity(genre_emb[movie_idx].reshape(1,-1),genre_emb[remm_idx].reshape(1,-1))[0,0]

    overview_sim = cosine_similarity(overview_emb[movie_idx].reshape(1,-1),overview_emb[remm_idx].reshape(1,-1))[0,0]

    keyword_sim = cosine_similarity(keyword_emb[movie_idx].reshape(1,-1),keyword_emb[remm_idx].reshape(1,-1))[0,0]

    director_sim = cosine_similarity(director_emb[movie_idx].reshape(1,-1),director_emb[remm_idx].reshape(1,-1))[0,0]

    final_score =(gen_sim*30 + overview_sim*35 + keyword_sim  *30 + director_sim *0.05)

    return {'genre_similarity' : gen_sim, 
            "keyword_similarity": keyword_sim,
        "overview_similarity": overview_sim,
        "director_similarity": director_sim,
        "final_score": final_score
           }
    

In [33]:
get_similarity_explaination(95,220)

{'genre_similarity': np.float32(0.91419053),
 'keyword_similarity': np.float32(0.28998202),
 'overview_similarity': np.float32(0.1413234),
 'director_similarity': np.float32(0.43053916),
 'final_score': np.float32(41.09302)}

In [34]:
overview_emb.shape

(3037, 384)

In [35]:
movies_df[movies_df['title'] == 'Prometheus']

,movie_id,budget,homepage,original_title,popularity,production_companies,production_countries,release_date,revenue,runtime,...,title,vote_average,vote_count,overview_clean,genres_clean,cast_clean,crew_clean,keywords_clean,original_language_full,combined_text
158,70981,130000000,http://www.projectprometheus.com/,Prometheus,68.889395,"[{""name"": ""Twentieth Century Fox Film Corporat...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2012-05-30,403170142,124.0,...,Prometheus,6.3,5080,A team of explorers discover a clue to the ori...,"Science Fiction, Mystery, Adventure","Noomi Rapace, Michael Fassbender, Guy Pearce,...",Director: Ridley Scott,"alien, stasis, creation, spin off, cave drawi...",English,"Genres : Science Fiction, Mystery, Adventure...."


In [36]:
cosine_similarity(genre_emb[1].reshape(1,-1),genre_emb[6].reshape(1,-1))[0][0]

np.float32(1.0)